# Train both models on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yuiandyuki/receipt-extraction-cord/blob/main/notebooks/train_colab.ipynb)

1. **Runtime → Change runtime type → L4 GPU** (or T4; an A100 only saves a few minutes here). A Google AI Pro plan includes monthly Colab compute units; check *Settings → Subscription* in Colab.
2. **Run all** and keep this tab open (paid plans other than Pro+/Ultra have no background execution).
3. The last cells copy `results/` and the best model to Google Drive and download `results.zip`.

Time per model is not measured on Colab yet (rough guess: 10–20 min on a T4, less on an L4). For reference, on an Apple M3 (MPS) training ran at ~0.9 steps/s (text) and ~0.6 steps/s (layout), i.e. ~30 and ~40 min for the 1,500 steps.

Everything the README reports comes from `results/`, so commit the downloaded `results/` back to the repo.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
![ -d receipt-extraction-cord ] || git clone -q https://github.com/yuiandyuki/receipt-extraction-cord.git
%cd receipt-extraction-cord
!git log -1 --oneline

Colab already ships a CUDA build of torch (and, as of Oct 2026, the same transformers / datasets / accelerate versions this repo pins), so keep its torch and install only what is missing. The torch version actually used is recorded in each `results/metrics/*.json`.

In [ ]:
!pip install -q seqeval==1.2.2 transformers==5.20.2 datasets==4.8.5 accelerate==1.15.0
!pip install -q --no-deps -e .
import torch, transformers
print(torch.__version__, transformers.__version__, torch.cuda.get_device_name(0))

## Train (same settings for both: 15 epochs, batch 8, lr 5e-5, seed 42)

The best epoch is chosen on validation; test is evaluated once at the end.

In [ ]:
!python -m receipt.train --model text --seed 42

In [ ]:
!python -m receipt.train --model layout --seed 42

## Rule baseline and statistics

In [ ]:
!python -m receipt.baseline
!python -m receipt.stats \
    --system text=results/predictions/text_seed42_test.json \
    --system layout=results/predictions/layout_seed42_test.json \
    --system baseline=results/predictions/baseline_test.json \
    --out results/stats.json

## Optional: two more seeds per model (about 4× the time above)

In [ ]:
# for seed in (13, 7):
#     !python -m receipt.train --model text --seed {seed}
#     !python -m receipt.train --model layout --seed {seed}

## Save results and the best model

Pick the model that did better on **validation** (not test) for the API. Weights are ~0.5–1 GB, so they go to Drive / a GitHub Release, never into git.

In [ ]:
import json, pathlib, shutil
val = {m: json.load(open(f'results/metrics/{m}_seed42.json'))['validation']['micro']['f1'] for m in ('text', 'layout')}
best = max(val, key=val.get)
print('validation micro F1:', val, '-> API model:', best)
shutil.make_archive('results', 'zip', '.', 'results')
shutil.make_archive(f'model-{best}', 'zip', f'runs/{best}_seed42/best')

from google.colab import drive
drive.mount('/content/drive')
dest = pathlib.Path('/content/drive/MyDrive/receipt-extraction-cord')
dest.mkdir(parents=True, exist_ok=True)
for f in ('results.zip', f'model-{best}.zip'):
    shutil.copy(f, dest / f)
print('copied to', dest)

In [ ]:
from google.colab import files
files.download('results.zip')